<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/06_validate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6 - Validate the way the client validates, then stress it

Reproduce the client's diagnostic (Pearson r between July annual means, n = years) for
both models on the same lakes and field data, then add the three things the client did
not: a bootstrap CI on r, a sensitivity grid over the analyst's discretionary choices,
and a provenance check of Water Quality Portal Secchi against LAGOS-US LIMNO.

The credibility anchor: the national model should fail on our Michigan lakes too.
Reproducing the client's failure on different lakes in a different state, before claiming
to fix it, proves the failure is structural rather than specific to Squam.

In [ ]:
# Cell 1 of 5: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

# Force a fresh import of the freshly pulled code. Colab keeps modules from a
# previous run in memory, so a git pull updates the files on disk while the
# kernel keeps running the old lakeclarity and silently ignores any fix. Purge
# them here so the import in the next cell always loads the pulled version.
for _stale in [m for m in list(sys.modules)
               if m == 'lakeclarity' or m.startswith('lakeclarity.')]:
    del sys.modules[_stale]


In [ ]:
# Cell 2 of 5: pull field Secchi from the Water Quality Portal REST API
import pandas as pd
from lakeclarity import config, locus, validate, viz, wqp

viz.use_style()
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]
lakes = locus.load_lakes().set_index("lagoslakeid")

# Confirm the characteristic-name choice before relying on it.
coverage = wqp.characteristic_coverage()
print(coverage.to_string(index=False))

field = wqp.fetch_secchi()  # cached to Drive after the first pull
# Attach lagoslakeid to every field reading via the Phase 2 site->lake map, so
# validation can filter field Secchi to a specific target lake.
site_to_lake = pd.read_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet")
field = field.merge(site_to_lake[["site_id", "lagoslakeid"]], on="site_id", how="left")
print(f"\n{len(field):,} Secchi records in the region, {field.year.min()}-{field.year.max()}")
print("mapped to a target lake:", field["lagoslakeid"].isin(target_ids).sum(), "records")

In [ ]:
# Cell 3 of 5: the like-for-like comparison, with bootstrap intervals
predictions = pd.read_parquet(config.PROCESSED_DIR / "predictions_full.parquet")
national = pd.read_parquet(config.INTERIM_DIR / "national_predictions_region.parquet")
national = national.rename(columns={"Secchi_predicted": "secchi_predicted_m"})
national["sensing_dt"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)
national["year"] = national["sensing_dt"].dt.year
national["month"] = national["sensing_dt"].dt.month

# Field Secchi per target lake, via the site->lake map attached in cell 2. If the
# client supplies their own station list, filter `field` to it here instead.
results = []
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    obs = field[field["lagoslakeid"] == lake_id]
    for label, pred in [("regional", predictions), ("national", national)]:
        res = validate.july_validation(pred, obs, lake_id, name, label)
        if res:
            results.append(res)
            print(res.sentence())

In [ ]:
# Cell 4 of 5: does the result survive the analyst's choices?
# Rebuild predictions under each configuration, then grid the headline r.
from itertools import product

configs = {}
for window, floor, qa, months in product([1, 3], [10, 25], ["on"], [(7,), (6, 7, 8)]):
    p = predictions[predictions["Pixelcount"] >= floor]
    configs[(window, floor, qa, months)] = p

for lake_id in target_ids:
    grid = validate.sensitivity_grid(configs, field, lake_id)
    grid.to_csv(config.TABLE_DIR / f"T13_sensitivity_{lake_id}.csv", index=False)
    print(f"\nlake {lake_id} ({lakes.loc[lake_id, 'lake_name']}):")
    print(grid.to_string(index=False))
    if grid["r"].notna().any():
        print(f"  r ranges [{grid['r'].min():+.2f}, {grid['r'].max():+.2f}] across the grid")

In [ ]:
# Cell 5 of 5: figures F25 to F28
matchups = pd.read_parquet(config.INTERIM_DIR / "matchups.parquet") if (config.INTERIM_DIR / "matchups.parquet").exists() else None

for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    print("wrote", viz.save(validate.fig_validation_overlay(
        field, predictions, national, lake_id, name), "F25", f"overlay_{lake_id}").name)
    grid = validate.sensitivity_grid(configs, field, lake_id)
    print("wrote", viz.save(validate.fig_sensitivity_grid(grid, name), "F27", f"sensitivity_{lake_id}").name)
    if matchups is not None:
        merged = validate.provenance_check(field, matchups, lake_id)
        if len(merged) > 2:
            print("wrote", viz.save(validate.fig_provenance(merged, name), "F28", f"provenance_{lake_id}").name)

print("wrote", viz.save(validate.fig_bootstrap_r(results), "F26", "bootstrap_r").name)